# Phenotype Data Preprocessing

## Description

This mini-protocol walks a new user through preparing a molecular phenotype matrix (for example, a bulk RNA-seq expression matrix) so that it is ready for QTL analysis with TensorQTL. It chains three modules together:

1. `gene_annotation.ipynb` (step i): adds genomic coordinates (`#chr`, `start`, `end`, `strand`) to each gene and converts the matrix to BED format.
2. `phenotype_imputation.ipynb` (step ii): fills in missing values. The recommended method is grouped Empirical Bayes Matrix Factorization (gEBMF).
3. `phenotype_formatting.ipynb` (step iii): splits the annotated matrix into one file per chromosome, which is what TensorQTL expects for cis analysis.

Estimated runtime for this toy example: under 12 minutes.

**Imputation methods available** (step ii): gEBMF (recommended), EBMF, missforest, knn, soft (SoftImpute), mean, lod, and bed_filter_na. Before imputing, the module applies QC filters that remove features with more than 40% missingness or more than 95% zero values (unless you pass `--no-qc-prior-to-impute`).

## Input Files

| File | Description |
|------|-------------|
| `input/rnaseq/protocol_example.rnaseq.bed.gz` | Bulk RNA-seq expression matrix (60 samples x 300 genes). First column is `gene_id`; remaining columns are samples. No genomic coordinates yet. |
| `input/reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf` | Collapsed gene-model GTF used to look up each gene's chromosome, start, end, and strand. |
| `input/protocol_example.protein.bed.gz` | Example protein phenotype matrix that contains missing values, used to demonstrate imputation. |

## Steps

### **Step 1.** Phenotype Annotation

This step adds `#chr`, `start`, `end`, `ID`, and `strand` columns to each gene in the matrix by matching gene IDs against the GTF. In the input, each column is a sample and each row is a gene. After annotation, the matrix is written in BED format ready for the next steps.

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
zcat output/rnaseq/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.bed.gz | head | cut -f 1-6

Inspect the annotated output (first six columns):

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/gene_annotation.ipynb annotate_coord \
    --cwd output/rnaseq \
    --phenoFile output/rnaseq/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.bed.gz \
    --coordinate-annotation reference_data/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.ERCC.gtf \
    --phenotype-id-column gene_id

```
gene_id SAMPLE_001      SAMPLE_002      SAMPLE_003      SAMPLE_004      SAMPLE_005
ENSG00000284070 4.46908614068004        3.6324384389436 0       0.926449274182614       8.44864560635242
ENSG00000273342 0       19.892726241898 13.6287618206549        28.5428919916226        10.1099209273507
ENSG00000128274 28.6051391056338        18.1638378952795        0       0.282770008563753       0
ENSG00000199515 6.2941881961677 14.0545732242658        0       0       12.4065609547843
ENSG00000279954 0       0       21.8339353514185        13.6383455478037        0
ENSG00000202058 16.8255985509548        4.33345932385917        0       17.118912521119 8.23661813672969
ENSG00000235271 8.88176894261571        6.76692962790534        20.3927068980862        0       0
ENSG00000235969 0       0.392169997712103       2.10063151606335        13.9548410786063        28.0335475000395
ENSG00000270083 28.1787728022632        6.28756003668852        0       8.02044591613714        7.59626520116873
```

### **Step 2.** Missing Value Imputation

This step fills in missing entries of the molecular phenotype data. It is optional for eQTL analysis, but necessary for many other QTL types. Here we use gEBMF (grouped Empirical Bayes Matrix Factorization), the recommended method. The example below uses a protein phenotype file that contains missing values, and skips QC prior to imputation.

**Input format.** The phenotype file has four leading columns `chr`, `start`, `end`, `ID`, followed by one column per sample. Missing values in the sample columns are what this step fills in.

**Quality control (optional).** By default, before imputing, the module drops features with more than 40% missing values or more than 95% zero values. Here we pass `--no-qc-prior-to-impute` to skip this filtering so the toy example keeps all features.

**Available methods.** You can choose any of eight algorithms via the workflow name:
- `gEBMF` (recommended): grouped Empirical Bayes Matrix Factorization. It partitions the data by chromosome, initializes clusters, runs backfitting iterations to fit the factor model, fills in the missing entries from that model, and applies an inverse-normal transform when values fall in the [0,1] range.
- `EBMF`: standard Empirical Bayes Matrix Factorization (ungrouped), using the flashier package.
- `missforest`: iterative random-forest imputation.
- `knn`: k-nearest-neighbors imputation.
- `soft`: SoftImpute via SVD.
- `mean`: replaces missing entries with the feature (row) mean.
- `lod`: limit-of-detection imputation.
- `bed_filter_na`: imputation combined with feature filtering.

**Tip.** `--num-factor` must be smaller than the number of samples. With 60 samples here we use `--num-factor 20`; `--num-factor` must be smaller than the sample count, so the default (60) would fail on this small toy dataset.

**Output.** A fully imputed matrix with the same structure as the input, written as `*.imputed.bed.gz` (bgzipped and tabix-indexed).

In [ ]:
zcat output/rnaseq/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.bed.bed.gz | head | cut -f 1-6

### **Step 3.** Partition by Chromosome

This step is required for cis TensorQTL analysis. For each chromosome (chr1-chr22), it produces a `chr#.bed.gz` file and its `chr#.bed.gz.tbi` index, plus a manifest file (`*_files.txt`) that lists the path for each chromosome. It uses the phenotype file produced by step i.

In [ ]:
# step ii. Missing Value Imputation
# This step serves as impute the missing entries for molecular phenotype data. This step is optional for eQTL analysis. But for other QTL analysis, this step is necessary. The missing entries are imputed by flashier, a Empirical Bayes Matrix Factorization model.

sos run pipeline/phenotype_imputation.ipynb gEBMF \
    --phenoFile data/protocol_example.protein.bed.gz \
    --cwd output/phenotype/impute_gebmf \
    --no-qc-prior-to-impute # skip QC before impupation

## Output Files

| File | Description |
|------|-------------|
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Coordinate-annotated phenotype BED (output of step i). |
| `output/phenotype/impute_gebmf/*.imputed.bed.gz` | Imputed phenotype matrix, same structure as the input (output of step ii). |
| `output/phenotype/phenotype_by_chrom/*.chr{1..22}.bed.gz` (with `.tbi` index) and `phenotype_by_chrom_files.txt` | Per-chromosome phenotype files and the manifest listing their paths (output of step iii). |

## Anticipated Results

Phenotype preprocessing produces a coordinate-annotated, optionally imputed, per-chromosome phenotype file set that is formatted and ready for use in TensorQTL.

In [11]:
#this uses results of phenotype file after it has been annotated with gene_annotation.ipynb annotate_coord
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/phenotype/phenotype_by_chrom_for_cis \
    --phenoFile output/rnaseq/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.bed.bed.gz \
    --name bulk_rnaseq \
    --chrom `for i in {1..22}; do echo chr$i; done`

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Running phenotype_by_chrom_1: 
INFO: phenotype_by_chrom_1 (index=1) is completed.
INFO: phenotype_by_chrom_1 (index=0) is completed.
INFO: phenotype_by_chrom_1 (index=2) is completed.
INFO: phenotype_by_chrom_1 (index=5) is completed.
INFO: phenotype_by_chrom_1 (index=3) is completed.
INFO: phenotype_by_chrom_1 (index=4) is completed.
INFO: phenotype_by_chrom_1 (index=6) is completed.
INFO: phenotype_by_chrom_1 (index=8) is completed.
INFO: phenotype_by_chrom_1 (index=7) is completed.
INFO: phenotype_by_chrom_1 (index=10) is completed.
INFO: phenotype_by_chrom_1 (index=9) is completed.
INFO: phenotype_by_chrom_1 (index=12) is completed.
INFO: